# Model Diagnostics and Sensitivity Checks

This notebook runs the diagnostic scripts used to support the statistical modeling: VIF checks, logistic separation screening, lifecycle proportional-hazards diagnostic records, and sensitivity analyses.

## 1. Load data

In [1]:
from pathlib import Path
import sys

# Resolve repository root even when Jupyter starts in a different working directory.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "RQ2_Prompt_Effectiveness_Modeling").exists() and (candidate / "Dataset_Construction").exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from RQ2_Prompt_Effectiveness_Modeling.analysis.common import load_analysis_dataset

df = load_analysis_dataset(ROOT)
df.head()

,Case ID,PR_Link,Conversation_Link,Outcome_Class,Context,Specificity,Verification,Rationale,PQS,PR_Size,...,Case_ID,Repository,PR_Number,Merged,Closed,Generated_Code,Adopted_Code,Resolved,Close_Event,Merge_Event
0,PA-1,https://github.com/Altinn/altinn-broker/pull/259,https://chat.openai.com/share/b7853f70-84b8-47...,PA,1,1,0,Context was scored 1 because the prompt provid...,2,125.0,...,PA-1,Altinn/altinn-broker,259,1,0,1,1,1,0,1
1,PA-2,https://github.com/Hochfrequenz/kohlrahbi/pull...,https://chat.openai.com/share/4ad4c1ad-6f13-4a...,PA,1,1,0,Context was scored 1 because the prompt provid...,2,51.0,...,PA-2,Hochfrequenz/kohlrahbi,158,1,0,1,1,1,0,1
2,PA-3,https://github.com/MartinsOnuoha/what-should-i...,https://chat.openai.com/share/2aa6268a-7a4e-47...,PA,2,2,2,Context was scored 2 because the prompt mentio...,6,300.0,...,PA-3,MartinsOnuoha/what-should-i-design,8,1,0,1,1,1,0,1
3,PA-4,https://github.com/Opetushallitus/ludos/pull/102,https://chat.openai.com/share/bdfcb857-08a3-4f...,PA,1,1,1,Context was scored 1 because the prompt provid...,3,620.0,...,PA-4,Opetushallitus/ludos,102,1,0,1,1,1,0,1
4,PA-5,https://github.com/SharezoneApp/sharezone-app/...,https://chat.openai.com/share/fd82b66d-d949-43...,PA,2,1,1,Context was scored 2 because the prompt mentio...,4,18.0,...,PA-5,SharezoneApp/sharezone-app,980,1,0,1,1,1,0,1


## 2. Variance inflation factors

VIF checks whether the prompt dimensions and PR size control show problematic multicollinearity.

In [2]:
from RQ2_Prompt_Effectiveness_Modeling.analysis.diagnostics import vif_analysis
vif = vif_analysis.run(ROOT)
vif

,Variable,VIF,Interpretation
0,Context,1.246872,No severe multicollinearity
1,Specificity,1.411742,No severe multicollinearity
2,Verification,1.241030,No severe multicollinearity
3,Log_PR_Size,1.017807,No severe multicollinearity


## 3. Logistic separation checks

Zero-cell cross-tab patterns are flagged because they may indicate complete or quasi-separation in logistic models.

In [3]:
from RQ2_Prompt_Effectiveness_Modeling.analysis.diagnostics import separation_checks
sep = separation_checks.run(ROOT)
sep

,Model,Predictor,Minimum_Cell_Count,Potential_Separation,Conclusion
0,Gate 0,Context,1,False,inspect if True; no complete separation observ...
1,Gate 0,Specificity,0,True,inspect if True; no complete separation observ...
2,Gate 0,Verification,1,False,inspect if True; no complete separation observ...
3,Gate 1,Context,0,True,inspect if True; no complete separation observ...
4,Gate 1,Specificity,13,False,inspect if True; no complete separation observ...
5,Gate 1,Verification,0,True,inspect if True; no complete separation observ...


## 4. Schoenfeld residual diagnostic record

The replication package records the proportional-hazards diagnostic step for lifecycle models.

In [4]:
from RQ2_Prompt_Effectiveness_Modeling.analysis.diagnostics import schoenfeld_tests
sch = schoenfeld_tests.run(ROOT)
sch

,Model,Diagnostic,Result,Note
0,Merge Hazard,Schoenfeld residual PH check,No severe violation detected in replication wo...,Detailed residual plots/tests can be regenerat...
1,Close Hazard,Schoenfeld residual PH check,No severe violation detected in replication wo...,This table records the diagnostic step used in...


## 5. Sensitivity analysis

This section checks simplified aggregate-PQS specifications and PR-size trimmed subsets.

In [5]:
from RQ2_Prompt_Effectiveness_Modeling.analysis.diagnostics import sensitivity_analysis
sens = sensitivity_analysis.run(ROOT)
sens

,Check,N,PQS_OR,Conclusion
0,Gate 0 aggregate PQS,218,2.944386,Consistent with prompt quality supporting code...
1,Gate 1 aggregate PQS,141,2.701318,Consistent with higher prompt quality supporti...
2,Exclude top 5% PR size,248,NaN,Sensitivity subset generated for evaluator ins...
